# Paraxial Optical-System Laboratory

Build an optical system from **translation**, **refraction**, **reflection**, **thin-lens**, and **thick-lens** components while keeping every matrix factor visible.

This notebook preserves the course convention

\[
\mathbf r=\begin{pmatrix}n\alpha\\x\end{pmatrix},\qquad
T=\begin{pmatrix}1&0\\d/n&1\end{pmatrix},\quad
R_a=\begin{pmatrix}1&-\mathcal P\\0&1\end{pmatrix},\quad
R_e=\begin{pmatrix}1&2n/R\\0&1\end{pmatrix}.
\]

Each thin or thick lens contributes **one** lens matrix to the product. The first component added is **rightmost** and multiplies the input ray first; later components pile on the left:

\[
M_{VV'}=M_n\cdots M_2 M_1.
\]

Because matrix multiplication is not commutative, reordering a component changes \(M_{VV'}\). Principal and conjugate planes are always calculated from that resolved base matrix.

In [27]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import sys

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np

import paraxial_config as config
import paraxial_engine as engine
import paraxial_tools as tools
from paraxial_style import ParaxialDashboard

print("Python:", sys.executable)
for module in (np, plt.matplotlib, widgets):
    print(f"  {module.__name__}: {module.__version__}")
print("Paraxial matrix, principal-plane, and conjugate-plane tools loaded OK.")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Python: d:\GitHub\Optics\.venv\Scripts\python.exe
  numpy: 2.4.6
  matplotlib: 3.11.0
  ipywidgets: 8.1.8
Paraxial matrix, principal-plane, and conjugate-plane tools loaded OK.


## 1. Source regression: lens matrices and product order

The following cell checks the original interface-power and translation examples, rebuilds the three thick lenses from the course radii, and verifies the MATLAB-written product

\[
M_t=M_1\,T\,M_2\,T\,M_3\approx\begin{pmatrix}0.5616&-0.0574\\11.9278&0.5622\end{pmatrix}.
\]

It then builds the interactive stack (first component rightmost) and prints that product order.

In [28]:
assert np.isclose(engine.interface_power(1.3, 1.0, 0.50), -0.6)
np.testing.assert_allclose(
    engine.translation_matrix(1.0, 1.5),
    [[1.0, 0.0], [1.5, 1.0]],
)

p1 = engine.interface_power(1.0, 1.812, 11.5)
p2 = engine.interface_power(1.812, 1.0, -127.0)
p3 = engine.interface_power(1.0, 1.695, -23.5)
p4 = engine.interface_power(1.695, 1.0, 10.2)
p5 = engine.interface_power(1.0, 1.812, 30.0)
p6 = engine.interface_power(1.812, 1.0, -15.0)

def matlab_thick(n, d, ps, pf):
    """Source MATLAB order: Ra(Ps)*T*Ra(Pf) with first factor leftmost."""
    return engine.cascade(
        (
            engine.refraction_matrix(ps),
            engine.translation_matrix(n, d),
            engine.refraction_matrix(pf),
        )
    )

m1 = matlab_thick(1.812, 5.00, p1, p2)
m2 = matlab_thick(1.695, 1.55, p3, p4)
m3 = matlab_thick(1.812, 5.00, p5, p6)
matlab_total = engine.cascade(
    (
        m1,
        engine.translation_matrix(1.0, 1.25),
        m2,
        engine.translation_matrix(1.0, 2.50),
        m3,
    )
)
expected_matlab = np.array([[0.5616, -0.0574], [11.9278, 0.5622]])
np.testing.assert_allclose(matlab_total, expected_matlab, atol=5e-5)
assert np.isclose(engine.assert_unit_determinant(matlab_total), 1.0)

# Interactive thick_lens_matrix lists optical order (Ps, T, Pf), then reverses once.
optical = (
    engine.refraction_matrix(p1),
    engine.translation_matrix(1.812, 5.00),
    engine.refraction_matrix(p2),
)
np.testing.assert_allclose(
    engine.thick_lens_matrix(1.812, 5.00, p1, p2),
    engine.cascade(reversed(optical)),
)

course_elements = tools.preset_elements("course_exercise")
course_system = tools.build_system(course_elements)
assert len(course_system.factors) == 5
assert course_system.product_expression.endswith("Mtk[L1]")
assert course_system.is_unit_determinant

print("MATLAB-written product M1 @ T @ M2 @ T @ M3 =")
print(matlab_total)
print("\nInteractive thick lens equals cascade(reversed(Ra_s, T, Ra_f)):")
print(engine.thick_lens_matrix(1.812, 5.00, p1, p2))
print("\nInteractive stack product (first component rightmost):")
print(course_system.product_expression)
for factor in course_system.factors:
    print(f"{factor.label} — {factor.description}")
    print(factor.matrix)
print("\nM_VV' =")
print(course_system.matrix)
print("det(M_VV') =", course_system.determinant)

AssertionError: 
Not equal to tolerance rtol=1e-07, atol=5e-05

Mismatched elements: 4 / 4 (100%)
Mismatch at indices:
 [0, 0]: 0.794807522088238 (ACTUAL), 0.5616 (DESIRED)
 [0, 1]: -0.05967340542080552 (ACTUAL), -0.0574 (DESIRED)
 [1, 0]: 9.901360751878785 (ACTUAL), 11.9278 (DESIRED)
 [1, 1]: 0.5147813455017379 (ACTUAL), 0.5622 (DESIRED)
Max absolute difference among violations: 2.02643925
Max relative difference among violations: 0.41525556
 ACTUAL: array([[ 0.794808, -0.059673],
       [ 9.901361,  0.514781]])
 DESIRED: array([[ 0.5616, -0.0574],
       [11.9278,  0.5622]])

## 2. Principal and conjugate plane sanity check

For a thin lens with \(\mathcal P=10\,\mathrm{m}^{-1}\), the next cell resolves

\[
T(V'\to H')\,M_{VV'}\,T(H\to V)=M_{HH'}
\]

and then uses an object distance \(s=0.2\,\mathrm m\) to build the complete object-to-image matrix. The assertions verify both the principal-plane reduction and the conjugate condition \(M_{21}=0\).

In [ ]:
lens_system = tools.build_system(
    [tools.OpticalElement.thin_lens("L1", 5.0, 5.0)]
)
cardinal = engine.principal_planes(lens_system.matrix)
conjugate = engine.conjugate_planes(cardinal, object_distance=0.2)

np.testing.assert_allclose(
    cardinal.reduction_matrix,
    cardinal.equivalent_matrix,
    atol=config.MATRIX_TOLERANCE,
)
assert conjugate.is_conjugate
assert np.isclose(conjugate.matrix[1, 0], 0.0, atol=config.MATRIX_TOLERANCE)
assert np.isclose(conjugate.lagrange_invariant, 1.0)

print("Principal reduction:", cardinal.principal_product_expression)
print(cardinal.reduction_matrix)
print(
    f"P={cardinal.power:.6g} m^-1, D={cardinal.object_principal_offset:.6g} m, "
    f"D'={cardinal.image_principal_offset:.6g} m"
)
print("\nConjugate product:", conjugate.product_expression)
print(conjugate.matrix)
print(
    f"s'={conjugate.image_distance:.6g} m, mx={conjugate.lateral_magnification:.6g}, "
    f"m_alpha={conjugate.angular_magnification:.6g}, M21={conjugate.conjugacy_residual:.3g}"
)

## 3. Interactive system builder

Use the controls below to:

1. load a preset or add translation, refraction, reflection, thin-lens, and thick-lens components;
2. move components with the arrow buttons and immediately see the non-commutative product change;
3. inspect each component’s single 2×2 matrix and the resolved vertex-to-vertex matrix \(M_{VV'}\) (first added = rightmost);
4. inspect the principal-plane values and the full reduction to \(M_{HH'}\);
5. choose an object distance from \(H\) and inspect the conjugate product, \(M_{21}=0\), magnifications, image classification, and ray schematic.

A thin or thick lens adds exactly one lens matrix to the product.

In [ ]:
dashboard = ParaxialDashboard()
dashboard.show()